# GPU Portfolio & Risk Decision Engine — Colab GPU runner

This notebook is **self-contained**: the full project source is embedded below, so you do not need GitHub or any local files.

## Before you Run All
1. **Runtime → Change runtime type → T4 GPU** (or better), then Save.
2. **Runtime → Run all.**

The notebook will: unpack the source, install the RAPIDS + cuOpt GPU stack, then run — **in this order, parity before speed** —
the test suite, the CPU/GPU numerical parity check, the benchmark sweep, and a backtest.

> On the **free T4 tier** system RAM (~12 GB) is under cuOpt's stated 16 GB minimum. Expect n=50 and n=500 to run fine and
> n=3000 to be tight or OOM — the sweep here uses 50/500 for that reason. Bump it up on a higher-RAM instance.

> These install commands are NVIDIA's current documented ones (cuOpt 26.02 / RAPIDS 26.06, CUDA 12). If NVIDIA has since
> moved the API, the parity cell is where it will surface — `optimizer/cuopt_compat.py` raises a message naming the docs page.


## 1. Confirm a GPU is attached

If this errors or shows no GPU, fix the runtime type (step 1 above) before continuing.


In [ ]:
import subprocess
out = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(out.stdout or out.stderr)
assert out.returncode == 0, 'No NVIDIA GPU visible — set Runtime → Change runtime type → T4 GPU, then Run all again.'

## 2. Unpack the embedded project source

The entire repo is stored as a base64 blob in the next cell and extracted into `/content/gpu-portfolio-optimization-engine`.


In [ ]:
import base64, io, tarfile, os

# base64 of a gzipped tar of the project (source only — no venv, caches, or git).
REPO_B64 = (
    "..."
)

PROJECT_DIR = '/content/gpu-portfolio-optimization-engine'
os.makedirs(PROJECT_DIR, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(''.join(REPO_B64))), mode='r:gz') as tar:
    tar.extractall(PROJECT_DIR)
os.chdir(PROJECT_DIR)
print('extracted into', PROJECT_DIR)
print(sorted(os.listdir('.')))

## 3. Install dependencies

CPU requirements first, then the GPU stack from NVIDIA's package index. **This cell takes several minutes** (the GPU wheels are large).
Pinned to the versions the code was written against: cuOpt 26.02, RAPIDS 26.06, CUDA 12.


In [ ]:
# CPU deps (fast).
!pip install -q -r requirements.txt

# GPU stack from NVIDIA's index. cu12 wheels to match Colab's CUDA 12 runtime.
!pip install -q --extra-index-url=https://pypi.nvidia.com \
    'cudf-cu12==26.6.*' 'cuml-cu12==26.6.*' 'cupy-cuda12x' 'cuopt-cu12==26.2.*'

## 4. Verify the GPU stack imports

If this prints versions, cuDF and cuOpt are installed and importable. If it fails, the parity/benchmark cells below will fall back to CPU and say so.


In [ ]:
import importlib
for m in ['cudf','cupy','cuml','cuopt']:
    try:
        mod = importlib.import_module(m)
        print(f'{m:8} ' + str(getattr(mod, '__version__', '?')))
    except Exception as e:
        print(f'{m:8} IMPORT FAILED: ' + type(e).__name__ + ': ' + str(e))

## 5. Test suite

GPU-only tests that were skipped on your Mac should now actually execute.


In [ ]:
!python -m pytest tests/ -q

## 6. Numerical parity — CPU vs GPU  *(the moment of truth)*

This is the step that must pass **before** any timing is trusted. It checks that the cuDF/CuPy risk model and the cuOpt QP
solver produce the same numbers as the pandas/CVXPY reference — covariance to ~1e-9, objective value to 1e-8. It also runs
the quadratic-convention probe and the closed-form ground-truth cases that have no solver on the reference side.


In [ ]:
!python -m pipeline.parity_tests --n 500 --days 2520

## 7. Benchmark sweep — CPU vs GPU per stage

Now that parity holds, the timings mean something. Warm-up is reported separately, each point is the median of 5 runs, and
host→device transfer is its own line. From the CPU-only run you already know **solve dominates** — that's the stage to watch.


In [ ]:
!python -m benchmarks.run_benchmarks --sizes 50 500 --days 2520 --runs 5

## 8. Backtest — solution-quality parity

Runs the identical strategy through both backends. The point is that the GPU is not just faster but lands on the *same*
Sharpe / equity curve. (Strategy losing to equal-weight 1/N is the expected result — see the README's limitations section.)


In [ ]:
!python -m backtest.run_backtest --source synthetic --n 500 --days 2520 --frequency QE

## 9. (Optional) Download the results

Grabs the CSVs and `environment.json` (GPU model, driver, every library version) so the numbers are reproducible off-Colab.


In [ ]:
from google.colab import files  # noqa
import shutil, os
if os.path.isdir('benchmarks/results'):
    shutil.make_archive('/content/benchmark_results', 'zip', 'benchmarks/results')
    files.download('/content/benchmark_results.zip')
else:
    print('no results directory yet — run the sweep cell first')